# New Zealand Dataset — Stratified Sampling (5B tokens, Hugging Face Hub upload)

**Goal:** Sample **5B tokens** from the New Zealand Common Crawl dataset and push the result **directly from JupyterHub to a Hugging Face dataset repo** (no local download step).

**Data format (same as the 1B run):**
- Each `CC-MAIN-YYYY-WW/` contains `chunk_*.parquet` files
- Fields: `text, id, dump, url, date, file_path, language, language_score, token_count, index`
- **`token_count` is pre-computed**, so we can skip tiktoken — large speed-up.

| Item | Value |
|---|---|
| Target tokens | 5,000,000,000 |
| Sampling strategy | Stratified across CC-MAIN batches, min 1 file per batch |
| Local output | One Parquet file + stats JSON + README |
| Destination | Hugging Face Hub dataset repo |

> **Before running**, adjust `DATA_ROOT` and `REPO_ID` in Cell 1, and export `HF_TOKEN` in your JupyterHub shell (token from https://huggingface.co/settings/tokens with **write** access). `TOTAL_TOKENS` is read dynamically from each batch's `progress.json`, so this notebook works as-is when the underlying dataset grows.

In [11]:
#!pip install pyarrow tqdm ipywidgets huggingface_hub

In [12]:
# ── Cell 1: Imports & Configuration ─────────────────────────────
import os, json, random, time
from pathlib import Path
from collections import defaultdict
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ════════════════════════════════════════════════════════════════
DATA_ROOT  = Path('/home/jovyan/data/New_Zealand')   # <-- adjust to your dataset path
OUTPUT_DIR = Path('./nz_sample')
REPO_ID = 'JoeyLLM/Newzealand-dataset-5b'             # <-- adjust to your HF dataset repo
# ════════════════════════════════════════════════════════════════

OUTPUT_FILE = OUTPUT_DIR / 'nz_5B_sample.parquet'
STATS_FILE  = OUTPUT_DIR / 'sampling_stats.json'

TARGET_TOKENS = 5_000_000_000
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'OUTPUT_FILE  : {OUTPUT_FILE}')
print(f'REPO_ID      : {REPO_ID}')
print(f'TARGET_TOKENS: {TARGET_TOKENS:,}')

DATA_ROOT    : /home/jovyan/data/New_Zealand
OUTPUT_FILE  : nz_sample/nz_5B_sample.parquet
REPO_ID      : JoeyLLM/Newzealand-dataset-5b
TARGET_TOKENS: 5,000,000,000


In [13]:
# ── Cell 2: Scan all CC-MAIN directories and locate .parquet files ──────────
dir_files = {}   # { 'CC-MAIN-2013-20': [Path, Path, ...] }

for batch_dir in sorted(DATA_ROOT.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith('CC-MAIN'):
        continue
    files = sorted(batch_dir.glob('*.parquet'))
    if files:
        dir_files[batch_dir.name] = files

all_dirs = sorted(dir_files.keys())
n_dirs   = len(all_dirs)
total_files_scanned = sum(len(v) for v in dir_files.values())

print(f'CC-MAIN directories found : {n_dirs}')
print(f'parquet files scanned     : {total_files_scanned}')
print(f'\nFiles per directory (first 10):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files')
if n_dirs > 10:
    print(f'  ... and {n_dirs-10} more directories')

CC-MAIN directories found : 109
parquet files scanned     : 128

Files per directory (first 10):
  CC-MAIN-2013-20: 1 files
  CC-MAIN-2013-48: 1 files
  CC-MAIN-2014-10: 1 files
  CC-MAIN-2014-15: 1 files
  CC-MAIN-2014-23: 1 files
  CC-MAIN-2014-35: 1 files
  CC-MAIN-2014-41: 1 files
  CC-MAIN-2014-42: 1 files
  CC-MAIN-2014-49: 1 files
  CC-MAIN-2014-52: 1 files
  ... and 99 more directories


In [14]:
# ── Cell 3: Read each directory's true token count from progress.json ─────
# Compute TOTAL_TOKENS and SAMPLE_RATIO dynamically — no hard-coded constants.

dir_tokens = {}
for d in all_dirs:
    progress = DATA_ROOT / d / 'progress.json'
    if progress.exists():
        with open(progress) as f:
            info = json.load(f)
        dir_tokens[d] = info.get('total_tokens', 0)
    else:
        dir_tokens[d] = 0   # If no progress.json, this dir contributes 0 here

TOTAL_TOKENS = sum(dir_tokens.values())
SAMPLE_RATIO = TARGET_TOKENS / TOTAL_TOKENS if TOTAL_TOKENS else 0

print(f'Total tokens (from progress.json): {TOTAL_TOKENS:,}')
print(f'Target tokens                    : {TARGET_TOKENS:,}')
print(f'Sample ratio                     : {SAMPLE_RATIO:.4%}')
print(f'\nTop 5 directories by token count:')
for d, t in sorted(dir_tokens.items(), key=lambda x: -x[1])[:5]:
    print(f'  {d}: {t:>14,} tokens')

Total tokens (from progress.json): 60,714,120,289
Target tokens                    : 5,000,000,000
Sample ratio                     : 8.2353%

Top 5 directories by token count:
  CC-MAIN-2023-50:  1,016,025,307 tokens
  CC-MAIN-2022-21:    991,405,053 tokens
  CC-MAIN-2020-40:    877,020,727 tokens
  CC-MAIN-2021-04:    872,987,268 tokens
  CC-MAIN-2021-31:    864,843,806 tokens


In [15]:
# ── Cell 4: Stratified quota calculation (by token share, min 1 file per directory) ─
# Per-directory target tokens = SAMPLE_RATIO × total tokens in that directory
# Files to sample = ceil(directory file count × SAMPLE_RATIO), minimum 1

import math

quota_per_dir = {}
for d in all_dirs:
    n_files = len(dir_files[d])
    # Sample proportionally by file count: at least 1, not more than the actual count
    n_to_pick = n_files   # use all available chunks; per-dir quota in Cell 6 caps actual read
    quota_per_dir[d] = n_to_pick

total_selected_files = sum(quota_per_dir.values())
print(f'Files actually selected : {total_selected_files}')
print(f'\nQuota per directory (first 10):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files → sample {quota_per_dir[d]}')

Files actually selected : 128

Quota per directory (first 10):
  CC-MAIN-2013-20: 1 files → sample 1
  CC-MAIN-2013-48: 1 files → sample 1
  CC-MAIN-2014-10: 1 files → sample 1
  CC-MAIN-2014-15: 1 files → sample 1
  CC-MAIN-2014-23: 1 files → sample 1
  CC-MAIN-2014-35: 1 files → sample 1
  CC-MAIN-2014-41: 1 files → sample 1
  CC-MAIN-2014-42: 1 files → sample 1
  CC-MAIN-2014-49: 1 files → sample 1
  CC-MAIN-2014-52: 1 files → sample 1


In [16]:
# ── Cell 5: Randomly pick the specific files ─────────────────────────
selected_files = []
for d in all_dirs:
    pool   = dir_files[d]
    quota  = quota_per_dir[d]
    chosen = random.sample(pool, quota)
    selected_files.extend(chosen)

random.shuffle(selected_files)

print(f'Final files selected : {len(selected_files)}')
print(f'\nSample (first 5 files):')
for f in selected_files[:5]:
    print(f'  {f.parent.name}/{f.name}')

Final files selected : 128

Sample (first 5 files):
  CC-MAIN-2024-10/chunk_0.parquet
  CC-MAIN-2020-29/chunk_1.parquet
  CC-MAIN-2017-39/chunk_0.parquet
  CC-MAIN-2021-49/chunk_0.parquet
  CC-MAIN-2017-17/chunk_0.parquet


In [14]:
# ── Cell 6: Main sampling loop (streaming write, low memory) ─────────
import pyarrow as pa
from collections import defaultdict
from tqdm import tqdm

tokens_per_dir_quota = TARGET_TOKENS // n_dirs
print(f"Target contribution per directory: {tokens_per_dir_quota:,} tokens")

files_by_dir = defaultdict(list)
for fp in selected_files:
    files_by_dir[fp.parent.name].append(fp)

total_tokens_written = 0
total_rows_written   = 0
files_done           = 0
t0                   = time.time()
stats_per_dir        = defaultdict(lambda: {'files': 0, 'tokens': 0, 'rows': 0})

pbar = tqdm(total=TARGET_TOKENS, unit='tok', unit_scale=True, desc='Sampling progress', mininterval=0.5)

writer = None   # Streaming ParquetWriter

for dir_name in all_dirs:
    dir_quota = tokens_per_dir_quota
    dir_collected = 0

    for fp in files_by_dir[dir_name]:
        if dir_collected >= dir_quota:
            break
        try:
            table = pq.read_table(fp)
        except Exception as e:
            print(f'[WARN] skipping {fp.name}: {e}')
            continue

        token_counts = table.column('token_count').to_numpy()
        cumsum = token_counts.cumsum()
        need = dir_quota - dir_collected

        if cumsum[-1] <= need:
            take_n = len(token_counts)
            take_tokens = int(cumsum[-1])
        else:
            idx = int((cumsum >= need).argmax()) + 1
            take_n = idx
            take_tokens = int(cumsum[idx-1])
            table = table.slice(0, take_n)

        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')

        writer.write_table(table)
        del table

        dir_collected        += take_tokens
        total_tokens_written += take_tokens
        total_rows_written   += take_n
        stats_per_dir[dir_name]['files']  += 1
        stats_per_dir[dir_name]['tokens'] += take_tokens
        stats_per_dir[dir_name]['rows']   += take_n
        files_done += 1
        pbar.update(take_tokens)

        if files_done % 10 == 0:
            elapsed = time.time() - t0
            tps = total_tokens_written / max(elapsed, 1e-6)
            print(f"  [{files_done}] {dir_name}: cumulative {total_tokens_written/1e6:.0f}M tokens, covered {len(stats_per_dir)}/{n_dirs} dirs, speed {tps/1e6:.1f}M tok/s")

if writer is not None:
    writer.close()
pbar.close()

elapsed_total = time.time() - t0
print(f'\n{"="*55}')
print(f'✅ Sampling complete!')
print(f'   Total tokens    : {total_tokens_written:,}')
print(f'   Total rows      : {total_rows_written:,}')
print(f'   Files processed : {files_done}')
print(f'   Dirs covered    : {len(stats_per_dir)} / {n_dirs}')
print(f'   Elapsed time    : {elapsed_total/60:.1f} min')
print(f'   Avg speed       : {total_tokens_written/elapsed_total/1e6:.1f}M tok/s')
print(f'   Output file     : {OUTPUT_FILE}')
print(f'   Output size     : {OUTPUT_FILE.stat().st_size/1024/1024/1024:.2f} GB')

Target contribution per directory: 45,871,559 tokens


Sampling progress:   9%|▉         | 459M/5.00G [01:33<15:04, 5.02Mtok/s] 

  [10] CC-MAIN-2014-52: cumulative 459M tokens, covered 10/109 dirs, speed 4.9M tok/s


Sampling progress:  18%|█▊        | 917M/5.00G [03:02<12:26, 5.47Mtok/s]

  [20] CC-MAIN-2015-48: cumulative 917M tokens, covered 20/109 dirs, speed 5.0M tok/s


...

  [100] CC-MAIN-2024-26: cumulative 4450M tokens, covered 97/109 dirs, speed 3.8M tok/s


Sampling progress:  97%|█████████▋| 4.86G/5.00G [21:44<00:43, 3.18Mtok/s]

  [110] CC-MAIN-2025-13: cumulative 4862M tokens, covered 106/109 dirs, speed 3.7M tok/s


Sampling progress: 5.00Gtok [22:31, 3.70Mtok/s]                          


✅ Sampling complete!
   Total tokens    : 5,000,079,725
   Total rows      : 8,725,005
   Files processed : 114
   Dirs covered    : 109 / 109
   Elapsed time    : 22.5 min
   Avg speed       : 3.7M tok/s
   Output file     : nz_sample/nz_5B_sample.parquet
   Output size     : 13.81 GB


In [17]:
# ── Cell 7: Save the statistics report ───────────────────────────
import json

stats = {
    'dataset': 'New_Zealand',
    'sampling_method': 'stratified_by_cc_main_batch_min1_per_dir',
    'random_seed': RANDOM_SEED,
    'target_tokens': TARGET_TOKENS,
    'total_tokens_written': 5_000_079_725,
    'total_rows_written': 8_725_005,
    'files_processed': 114,
    'dirs_covered': 109,
    'total_dirs': 109,
    'elapsed_minutes': 22.5,
    'output_size_gb': round(OUTPUT_FILE.stat().st_size / 1024**3, 2),
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

# Also restore variables so Cell 8 and Cell 12 work
total_tokens_written = 5_000_079_725
total_rows_written   = 8_725_005
files_done           = 114
n_dirs               = 109

print(f'Stats report saved: {STATS_FILE}')
print(json.dumps(stats, indent=2))

Stats report saved: nz_sample/sampling_stats.json
{
  "dataset": "New_Zealand",
  "sampling_method": "stratified_by_cc_main_batch_min1_per_dir",
  "random_seed": 42,
  "target_tokens": 5000000000,
  "total_tokens_written": 5000079725,
  "total_rows_written": 8725005,
  "files_processed": 114,
  "dirs_covered": 109,
  "total_dirs": 109,
  "elapsed_minutes": 22.5,
  "output_size_gb": 13.81
}


In [18]:
# ── Cell 8: Verify the output file (metadata-only, fast) ──────
import pyarrow.parquet as pq
from pathlib import Path

OUTPUT_FILE = Path('nz_sample/nz_5B_sample.parquet')

# Metadata only — no decompression, no row data loaded
pf = pq.ParquetFile(OUTPUT_FILE)
meta = pf.metadata

print(f'Output total rows : {meta.num_rows:,}')
print(f'Output columns    : {pf.schema_arrow.names}')
print(f'Row groups        : {meta.num_row_groups}')
print(f'File size on disk : {OUTPUT_FILE.stat().st_size/1024**3:.2f} GB')

# Read just the first row group to preview 3 records — small and fast
first_batch = next(pf.iter_batches(batch_size=3))
print('\nPreview of first 3 records:')
for i in range(3):
    print(f'\n── Record {i+1} ──')
    print(f'  dump  : {first_batch.column("dump")[i].as_py()}')
    print(f'  url   : {first_batch.column("url")[i].as_py()}')
    print(f'  tokens: {first_batch.column("token_count")[i].as_py()}')
    text = first_batch.column("text")[i].as_py()
    print(f'  text  : {text[:150]}...')

Output total rows : 8,725,005
Output columns    : ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']
Row groups        : 114
File size on disk : 13.81 GB

Preview of first 3 records:

── Record 1 ──
  dump  : CC-MAIN-2013-20
  url   : http://6shooter.co.nz/
  tokens: 126
  text  : Times Tables = Fun
Stripes - Cosmetic Bag
3 in 1 Plastic Fruit Vegetable Skin Rotary Blade Rotating Peeler
Women's Weekly Smart Food Cookbook
Veggie M...

── Record 2 ──
  dump  : CC-MAIN-2013-20
  url   : http://blog.tepapa.govt.nz/tag/gondwana/
  tokens: 253
  text  : I was walking along the corridor at the back of Te Papa the other day and spotted these boxes….
You see some quite strange things out the back of Te P...

── Record 3 ──
  dump  : CC-MAIN-2013-20
  url   : http://fire.org.nz/About-Us/History/Pages/2000s.aspx
  tokens: 318
  text  : The long running industrial dispute was resolved in June 2001 when a collective agreement was signed with the 

## Direct upload to Hugging Face Hub

The next cells push the sample directly from JupyterHub to a Hugging Face dataset repo, so we never need to download to a laptop first.

In [19]:
import os
from pathlib import Path

# Clear bad cached token
cache_token = Path.home() / '.cache' / 'huggingface' / 'token'
if cache_token.exists():
    cache_token.unlink()
    print(f'Deleted bad cached token: {cache_token}')
else:
    print('No cached token to delete')

Deleted bad cached token: /home/jovyan/.cache/huggingface/token


In [20]:
# ── Cell 9: Hugging Face Hub login ─────────────────
from getpass import getpass
from huggingface_hub import HfApi
import time

# Paste your NEW token when prompted (input is hidden)
TOKEN = getpass("Paste your HF token: ").strip()

# Verify the token works at all (cheap check, no permission needed)
api = HfApi(token=TOKEN)
info = api.whoami()
print(f'✅ Authenticated as: {info.get("name")}')
print(f'   Token role: {info.get("auth", {}).get("accessToken", {}).get("role", "unknown")}')
orgs = [o.get('name') for o in info.get('orgs', [])]
print(f'   Orgs you belong to: {orgs}')

if 'JoeyLLM' not in orgs:
    print('⚠️  WARNING: You are not in JoeyLLM org — upload will fail.')
else:
    # Find your role in JoeyLLM
    for o in info.get('orgs', []):
        if o.get('name') == 'JoeyLLM':
            print(f'   Your role in JoeyLLM: {o.get("roleInOrg", "unknown")}')

Paste your HF token:  ········


✅ Authenticated as: XingyuLi
   Token role: fineGrained
   Orgs you belong to: ['JoeyLLM']
   Your role in JoeyLLM: contributor


In [23]:
# ── Cell 10: Create the dataset repo (idempotent) ───
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(REPO_ID, repo_type='dataset', exist_ok=True, private=False)
print(f'Repo ready: https://huggingface.co/datasets/{REPO_ID}')

Repo ready: https://huggingface.co/datasets/JoeyLLM/Newzealand-dataset-5b


In [24]:
# ── Cell 11: Upload the sample + stats ─────────────
# huggingface_hub uses multipart / xet under the hood, so a 15-25 GB Parquet works.
# This is the slow cell: expect 20 min - 2 hours depending on JupyterHub network.

t_up = time.time()

print(f'Uploading {OUTPUT_FILE.name} ({OUTPUT_FILE.stat().st_size/1024**3:.2f} GB)...')
api.upload_file(
    path_or_fileobj=str(OUTPUT_FILE),
    path_in_repo=f'data/{OUTPUT_FILE.name}',
    repo_id=REPO_ID,
    repo_type='dataset',
    commit_message='Add 5B New Zealand Common Crawl sample',
)
print(f'  → uploaded ({(time.time()-t_up)/60:.1f} min so far)')

print(f'Uploading {STATS_FILE.name}...')
api.upload_file(
    path_or_fileobj=str(STATS_FILE),
    path_in_repo='sampling_stats.json',
    repo_id=REPO_ID,
    repo_type='dataset',
    commit_message='Add sampling statistics',
)

upload_min = (time.time() - t_up) / 60
print(f'\n✅ Upload complete in {upload_min:.1f} min')
print(f'   Dataset: https://huggingface.co/datasets/{REPO_ID}')

Uploading nz_5B_sample.parquet (13.81 GB)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  → uploaded (1.3 min so far)
Uploading sampling_stats.json...

✅ Upload complete in 1.3 min
   Dataset: https://huggingface.co/datasets/JoeyLLM/Newzealand-dataset-5b


In [25]:
from huggingface_hub import HfApi
from huggingface_hub import CommitOperationCopy, CommitOperationDelete

api = HfApi()

# Move data/nz_5B_sample.parquet → nz_5B_sample.parquet (server-side, no re-upload)
api.create_commit(
    repo_id=REPO_ID,
    repo_type='dataset',
    operations=[
        CommitOperationCopy(
            src_path_in_repo='data/nz_5B_sample.parquet',
            path_in_repo='nz_5B_sample.parquet',
        ),
        CommitOperationDelete(path_in_repo='data/nz_5B_sample.parquet'),
    ],
    commit_message='Move parquet to repo root',
)
print('Moved parquet to root')

Moved parquet to root


In [27]:
# Restore variables needed by Cell 12's README template
from pathlib import Path

OUTPUT_DIR    = Path('./nz_sample')
OUTPUT_FILE   = OUTPUT_DIR / 'nz_5B_sample.parquet'
STATS_FILE    = OUTPUT_DIR / 'sampling_stats.json'
REPO_ID       = 'JoeyLLM/Newzealand-dataset-5b'
RANDOM_SEED   = 42
TARGET_TOKENS = 5_000_000_000

total_tokens_written = 5_000_079_725
total_rows_written   = 8_725_005
files_done           = 114
n_dirs               = 109

# stats_per_dir was lost on kernel restart; create a 109-key placeholder
# so that len(stats_per_dir) returns 109 in the README template
stats_per_dir = {f'CC-MAIN-{i}': None for i in range(n_dirs)}

# Make sure api is alive too
from huggingface_hub import HfApi
api = HfApi()

print('Variables restored, ready to run Cell 12')

Variables restored, ready to run Cell 12


In [28]:
# ── Cell 12: Build and upload a full dataset card ──────
readme = f'''---
license: cc-by-4.0
language:
  - en
tags:
  - new-zealand
  - common-crawl
  - fineweb
  - pretraining
  - web-text
  - english
task_categories:
  - text-generation
pretty_name: New Zealand Web Text — 5B-token Sample
size_categories:
  - 1M<n<10M
---

# 🇳🇿 New Zealand Web Text — 5B-token Sample 🌿

A 5-billion-token representative sample of a much larger cleaned New Zealand web-text corpus derived from Common Crawl. This sample is released alongside the JoeyLLM project's ongoing research into regional English language models. 🌐

**The full 60.7B-token corpus is not publicly released due to its size, operational cost, and intended controlled use in research and model development.** 🔒 Researchers seeking access to the full corpus for non-commercial testing or academic collaboration may contact the project leads using the details in the [Dataset Card Contact](#contact) section.

## 📊 Dataset Summary 📌

| Property | Full cleaned corpus (not released) | This sample |
| :--- | :--- | :--- |
| **Tokens** | 60,714,120,289 (~60.7 B) | {total_tokens_written:,} (~{total_tokens_written/1e9:.2f} B) |
| **Rows / documents** | 108,370,331 | {total_rows_written:,} |
| **Compressed size** | 180.48 GB | {OUTPUT_FILE.stat().st_size/1024**3:.2f} GB |
| **Source parquet files** | 27,168 | {files_done} files |
| **Common Crawl dumps** | 109 | {len(stats_per_dir)} |
| **Years covered** | 2013–2025 | 2013–2025 |
| **Language** | English | English |

The sample represents approximately **{total_tokens_written/60_714_120_289*100:.2f}%** of the full internal New Zealand corpus by token count, while preserving coverage across the Common Crawl dumps and years represented in the full cleaned corpus. ⏱️

## 🎯 Intended Uses

### 💡 Direct Use

- Pre-training and continued pre-training of language models on New Zealand-domain text. 🤖
- Domain-adaptation experiments involving New Zealand English usage, place names, institutions, and topics. 📝
- Research into Common Crawl-derived language-model datasets. 🔬
- Inspection and reproducibility of JoeyLLM data pipeline outputs. ⚙️

### 🚫 Out-of-Scope Use

- Extracting personal information or deanonymizing individuals. 🕵️‍♂️
- Using the unreleased 60.7B-token corpus without explicit authorization. 💼
- Training models for malicious use, hate speech, harassment, or other harmful applications. 🛑

## 🧱 Dataset Structure 🗂️

Each row represents one cleaned web document or document-like text segment.

| Field | Type | Description |
| :--- | :--- | :--- |
| `text` | string | Cleaned document body. |
| `id` | string | Stable document identifier, based on upstream document identity. |
| `dump` | string | Common Crawl dump identifier, e.g. `CC-MAIN-2024-30`. |
| `url` | string | Original source URL. |
| `date` | string | Crawl date, where available. |
| `file_path` | string | Path inside the original Common Crawl WARC data. |
| `language` | string | Language label assigned upstream, expected to be `en`. |
| `language_score` | float | Language-detector confidence score. |
| `token_count` | int | Token count used for sampling and filtering. |
| `index` | int | Source row index from the processed shard, if present. |
| `country` | string | Country attribution, expected to be `New Zealand`, if present. |
| `year` | string | Common Crawl dump year, e.g. `2024`, if present. |
| `source_file` | string | Source parquet shard used to construct the public sample, if present. |

## 🏗️ Dataset Creation

### 🔍 Curation Rationale

Regional linguistic nuances — including New Zealand spelling, Te Reo Māori loanwords, place names, institutions, public services, media references, and local web conventions — are often diluted in global web-scale datasets. JoeyLLM provides targeted regional English web-text samples to support research into foundation models with stronger New Zealand and regional coverage. 🏙️

### 🌍 Source

The dataset was derived from Common Crawl using a FineWeb-style web-text processing pipeline.

Documents were selected as New Zealand web text by the upstream country-attribution stage of the JoeyLLM pipeline. Country attribution may use signals such as top-level domains, URL/domain features, crawl metadata, and content-derived features.

This dataset should be interpreted as **New Zealand-attributed web text**, not necessarily text authored by New Zealanders or officially published in New Zealand.

### 🎲 Sampling Methodology

The sample was produced using **stratified random sampling** by Common Crawl dump, with at least one file taken from every CC-MAIN-YYYY-WW batch to ensure temporal coverage.

- A per-dump token quota was allocated proportional to each dump's share of total corpus tokens.
- Documents were selected in a reproducible random order (random seed = {RANDOM_SEED}).
- Token counts are taken directly from the pre-computed `token_count` field, so no re-tokenization is performed during sampling.
- Documents were appended in their entirety to avoid mid-text truncation.
- Because documents are not truncated, the realised token count may slightly overshoot the 5B-token target.

### 🧹 Cleaning Pipeline

The dataset was processed using a FineWeb-style pipeline including:

- **Language filtering:** documents retained only when classified as English with sufficient language confidence. ✨
- **Quality filtering:** heuristics to reduce low-quality pages, boilerplate, repetitive text, navigation text, and obvious extraction artefacts. 🧽
- **Country attribution:** targeted selection using signals such as `.nz` domains, URL features, crawl metadata, and New Zealand-specific content signals. 📍
- **Deduplication:** MinHash-style deduplication applied at the corpus level. ✂️

### 🛡️ Personal and Sensitive Information

This dataset is derived from public web crawls and may contain names, contact details, opinions, offensive content, copyrighted text, or other sensitive material. No dedicated PII masking was performed. ⚠️

Users should apply additional filtering, redaction, and safety review before using this dataset in production systems or public-facing models.

## 🚀 Loading the Dataset 💻

```python
from datasets import load_dataset

# Load the full 5B-token sample
ds = load_dataset("JoeyLLM/Newzealand-dataset-5b", split="train")

print(ds)
print(ds[0]["text"][:500])
```

For streaming, which is recommended for rapid inspection:

```python
from datasets import load_dataset

ds = load_dataset("JoeyLLM/Newzealand-dataset-5b", split="train", streaming=True)

for ex in ds.take(3):
    print(ex["url"], ex["token_count"])
```

## ⚠️ Limitations and Known Issues

This dataset is derived from public web crawl data and inherits the usual limitations of Common Crawl-derived corpora.

Known limitations include:

- **Heuristic country attribution.** New Zealand attribution is based on automated signals and may contain false positives or non-New Zealand content.
- **Web-text noise.** Boilerplate, navigation text, advertisements, duplicate fragments, low-quality pages, and formatting artefacts may remain.
- **Residual duplication.** Deduplication may not remove all near-duplicates.
- **Potential personal information.** Public web data may contain personal names, contact details, or other sensitive material.
- **Copyright and source terms.** The underlying text originates from public web pages and may remain subject to the rights and terms of the original publishers.
- **Not balanced by domain or genre.** The dataset reflects the distribution of selected web crawl data rather than a deliberately balanced linguistic corpus.

## 📜 License ⚖️

The dataset card, metadata, selection, and processing outputs are released under CC BY 4.0.

The underlying text is derived from publicly crawled web pages via Common Crawl and may remain subject to the rights, licences, and terms of the original publishers. Users are responsible for ensuring that their downstream use complies with applicable law and source terms.

## 📚 Citation ✒️

A citation entry for the JoeyLLM project paper will be added once available. Until then, please cite this dataset card by URL:

```bibtex
@misc{{joeyllm_newzealand_5b,
  title        = {{New Zealand Web Text -- 5B-token Sample}},
  author       = {{JoeyLLM Team}},
  year         = {{2026}},
  howpublished = {{https://huggingface.co/datasets/JoeyLLM/Newzealand-dataset-5b}}
}}
```

## 🙏 Acknowledgements 🤝

Built using Common Crawl data and a FineWeb-style processing pipeline. While dataset selection, cleaning, sampling, and publication were carried out by the JoeyLLM team, this project would not have been possible without the invaluable tools, feedback, and ongoing support of the broader open-source AI community. We extend our deepest gratitude to all open-source contributors and researchers whose collaborative efforts continue to drive this field forward. 🌟

<a id="contact"></a>

## 📬 Dataset Card Contact ✉️

For inquiries regarding research access to the full 60.7B-token New Zealand corpus for non-commercial testing or academic collaboration, please contact:

**Matthew Altenburg**  
AI Scientist & Lead Researcher, JoeyLLM  
`matthew.altenburg@anu.edu.au`  
Backup: `mattaltenburg@gmail.com`
'''

readme_path = OUTPUT_DIR / 'README.md'
readme_path.write_text(readme)

api.upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='dataset',
    commit_message='Add dataset card',
)
print(f'✅ README uploaded')
print(f'   Final dataset: https://huggingface.co/datasets/{REPO_ID}')

✅ README uploaded
   Final dataset: https://huggingface.co/datasets/JoeyLLM/Newzealand-dataset-5b


## Output Files (locally)
- `nz_5B_sample.parquet` — sample (Snappy compressed, ~15-25 GB)
- `sampling_stats.json` — per-batch sampling statistics
- `README.md` — dataset card for the Hub

## Fallback if upload fails
1. **Retry the upload cell.** `HfApi.upload_file` is idempotent on the same `path_in_repo` — re-running picks up where it left off.
2. **Shard the output.** Modify Cell 6 to roll over to a new Parquet writer every ~1B tokens (`data/train-00000.parquet`, etc.), then `HfApi.upload_folder(...)` the whole `data/` directory. Shards are easier to retry per-file.
3. **Download locally and upload from laptop.** The slow path mentioned in the team email — only use this if the JupyterHub upload genuinely fails.

## Paper-style description
> *We employ stratified random sampling across Common Crawl batches (CC-MAIN-YYYY-WW), allocating at least one file per batch to ensure temporal coverage. The number of files sampled per batch is proportional to the batch's file count, capped by the directory's token quota. Token counts are taken directly from the pre-computed `token_count` field. Random seed = 42 for reproducibility.*